# Rescue Model — Recover Rejected Signals

The meta-confidence model (notebook 04) is a **precision filter**: it selects high-confidence trades (P > 0.55) with ~70% WR. But many rejected signals (P < 0.55 or |Q50| < 0.5x spread) are actually correct — the meta-model just can't distinguish them.

**Goal:** Train a "rescue model" on the rejected population to identify winners the current pipeline misses. This is **additive** — it doesn't touch the existing P > 0.55 trades.

**New contextual features** the meta-model doesn't see:
- Pair identity (label-encoded)
- Hour-of-day, day-of-week
- Cross-pair signal agreement (how many pairs signal same direction)
- Meta-model probability (as a continuous feature, not just a threshold)

**Models saved to:** `backend/models_5.1/rescue/`

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib
import warnings
import gc
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import roc_auc_score

FEATURES_DIR_MAJORS = Path('../backend/data/features_2')
FEATURES_DIR_CROSSES = Path('../backend/data/features_3')
MODELS_DIR   = Path('../backend/models_5.1/3_quants')
META_DIR     = Path('../backend/models_5.1/meta')
RESCUE_DIR   = Path('../backend/models_5.1/rescue')
RESCUE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END = '2024-06-30'
AVG_SPREAD = 0.00028
MIN_Q50_THRESHOLD = AVG_SPREAD * 0.5
META_THRESHOLD = 0.55  # current production threshold

print('Setup complete.')

## 1. Load Data & Regenerate OOF Predictions

Same as notebook 04 — we need OOF Q25/Q50/Q75 predictions + meta-model probabilities on training data.

In [ ]:
# Load dataset from BOTH feature dirs (15 pairs total)
df_majors = pd.read_parquet(FEATURES_DIR_MAJORS / 'all_pairs_microstructure.parquet')
df_crosses = pd.read_parquet(FEATURES_DIR_CROSSES / 'all_pairs_microstructure.parquet')

print(f'Majors:  {df_majors.shape} — pairs: {df_majors["pair"].unique()}')
print(f'Crosses: {df_crosses.shape} — pairs: {df_crosses["pair"].unique()}')

df = pd.concat([df_majors, df_crosses]).sort_index()
del df_majors, df_crosses

label_cols   = [c for c in df.columns if c.startswith('label_')]
drop_cols    = label_cols + ['pair']
feature_cols = [c for c in df.columns if c not in drop_cols]
TARGET_COL = 'label_1H'

df_train = df[df.index <= TRAIN_END].copy()
df_test  = df[df.index > TRAIN_END].copy()

# Clean training data
y_all = df_train[TARGET_COL]
valid_mask = y_all.notna()
X_clean = df_train[feature_cols][valid_mask].ffill().fillna(0)
y_clean = y_all[valid_mask]
pairs_clean = df_train.loc[valid_mask, 'pair']

print(f'\nTrain: {len(X_clean):,} rows | Test: {len(df_test):,} rows')
print(f'Features: {len(feature_cols)}')
print(f'All pairs: {sorted(df["pair"].unique())}')

# Walk-forward splits (same as training notebook)
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    test_size = int(n * test_ratio)
    splits = []
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n:
            break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

def get_lgbm_params(quantile):
    return {
        'objective': 'quantile', 'alpha': quantile, 'metric': 'quantile',
        'boosting_type': 'gbdt', 'n_estimators': 5000, 'learning_rate': 0.02,
        'num_leaves': 64, 'max_depth': 6, 'min_child_samples': 50,
        'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 5,
        'reg_alpha': 0.1, 'reg_lambda': 0.1, 'random_state': 42,
        'n_jobs': -1, 'verbose': -1, 'device': 'gpu',
    }

In [ ]:
# Regenerate OOF predictions for Q25, Q50, Q75
splits = walk_forward_splits(len(X_clean))
QUANTILES = [0.25, 0.50, 0.75]
QUANTILE_NAMES = ['Q25', 'Q50', 'Q75']

oof_preds = {q: np.full(len(X_clean), np.nan) for q in QUANTILES}

for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
    print(f'Generating OOF for {q_name}...')
    params = get_lgbm_params(q)
    
    for fold, (train_idx, test_idx) in enumerate(splits):
        X_tr, y_tr = X_clean.iloc[train_idx], y_clean.iloc[train_idx]
        X_te, y_te = X_clean.iloc[test_idx], y_clean.iloc[test_idx]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        
        oof_preds[q][test_idx] = model.predict(X_te)
    
    del model; gc.collect()
    valid_oof = ~np.isnan(oof_preds[q])
    print(f'  {q_name}: {valid_oof.sum():,} OOF predictions')

has_oof = ~np.isnan(oof_preds[0.50])
print(f'\nTotal OOF rows: {has_oof.sum():,} / {len(X_clean):,}')

In [ ]:
# Build full OOF dataframe with quantile predictions + actuals
oof_df = X_clean[has_oof].copy()
oof_df['Q25_oof'] = oof_preds[0.25][has_oof]
oof_df['Q50_oof'] = oof_preds[0.50][has_oof]
oof_df['Q75_oof'] = oof_preds[0.75][has_oof]
oof_df['abs_Q50'] = np.abs(oof_df['Q50_oof'])
oof_df['iqr'] = oof_df['Q75_oof'] - oof_df['Q25_oof']
oof_df['conf_ratio'] = oof_df['abs_Q50'] / oof_df['iqr'].clip(lower=1e-10)
oof_df['q50_dir'] = np.sign(oof_df['Q50_oof'])
oof_df['actual_return'] = y_clean.values[has_oof]
oof_df['actual_dir'] = np.sign(oof_df['actual_return'])
oof_df['pair'] = pairs_clean.values[has_oof]
oof_df['q50_correct'] = (oof_df['q50_dir'] == oof_df['actual_dir']).astype(int)

# Reset index to avoid duplicate datetime index (multiple pairs per timestamp)
oof_df = oof_df.reset_index()

# Generate OOF meta-model probabilities using walk-forward CV
# (same approach as notebook 04 — no leakage)
meta_feature_cols = feature_cols + ['Q50_oof', 'Q25_oof', 'Q75_oof', 'abs_Q50', 'iqr', 'conf_ratio']

tradeable = oof_df[oof_df['abs_Q50'] > MIN_Q50_THRESHOLD].copy()
X_meta = tradeable[meta_feature_cols]
y_meta = tradeable['q50_correct']

meta_params = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'n_estimators': 3000, 'learning_rate': 0.01, 'num_leaves': 32,
    'max_depth': 4, 'min_child_samples': 30, 'feature_fraction': 0.6,
    'bagging_fraction': 0.7, 'bagging_freq': 5, 'reg_alpha': 0.5,
    'reg_lambda': 0.5, 'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    'device': 'gpu', 'is_unbalance': True,
}

meta_splits = walk_forward_splits(len(X_meta), n_splits=5, test_ratio=0.1)
meta_oof_proba = np.full(len(X_meta), np.nan)

print(f'Generating OOF meta probabilities on {len(X_meta):,} tradeable rows...')
for fold, (train_idx, test_idx) in enumerate(meta_splits):
    X_tr, y_tr = X_meta.iloc[train_idx], y_meta.iloc[train_idx]
    X_te, y_te = X_meta.iloc[test_idx], y_meta.iloc[test_idx]
    
    model = lgb.LGBMClassifier(**meta_params)
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    
    meta_oof_proba[test_idx] = model.predict_proba(X_te)[:, 1]

del model; gc.collect()

# Assign meta probabilities back using integer positions (no index issues)
has_meta = ~np.isnan(meta_oof_proba)
oof_df['meta_proba'] = np.nan
oof_df.loc[tradeable.index[has_meta], 'meta_proba'] = meta_oof_proba[has_meta]

print(f'OOF rows with meta predictions: {has_meta.sum():,}')
print(f'Full OOF dataset: {len(oof_df):,} rows')

## 2. Build Rejected-Signal Dataset

The "rejected" population = signals the current pipeline would NOT trade:
- **Case A**: |Q50| > 0.5x spread BUT meta_proba < 0.55 (meta-model says no)
- **Case B**: |Q50| < 0.5x spread (never even reaches meta-model)

We add contextual features the meta-model is blind to.

In [ ]:
# Identify rejected signals — everything the current pipeline would NOT trade
# "Accepted" = |Q50| > 0.5x spread AND meta_proba > 0.55
accepted_mask = (
    (oof_df['abs_Q50'] > MIN_Q50_THRESHOLD) &
    (oof_df['meta_proba'] > META_THRESHOLD)
)
rejected_mask = ~accepted_mask & oof_df['meta_proba'].notna() | (oof_df['abs_Q50'] <= MIN_Q50_THRESHOLD)

# We need OOF meta probabilities for the rejected set.
# For Case B (|Q50| < 0.5x), meta_proba is NaN — we set it to 0 (never scored)
rejected = oof_df[rejected_mask].copy()
rejected['meta_proba'] = rejected['meta_proba'].fillna(0)

print(f'Total OOF rows: {len(oof_df):,}')
print(f'Accepted (would trade): {accepted_mask.sum():,}')
print(f'Rejected (rescue candidates): {len(rejected):,}')
print(f'  Case A (tradeable but meta < {META_THRESHOLD}): {((oof_df["abs_Q50"] > MIN_Q50_THRESHOLD) & (oof_df["meta_proba"].notna()) & (oof_df["meta_proba"] <= META_THRESHOLD)).sum():,}')
print(f'  Case B (|Q50| < 0.5x spread): {(oof_df["abs_Q50"] <= MIN_Q50_THRESHOLD).sum():,}')
print(f'\nRejected Q50 accuracy: {rejected["q50_correct"].mean():.1%}')
print(f'  (these are the signals we want to sift through)')
print(f'Class balance: {rejected["q50_correct"].value_counts().to_dict()}')

In [ ]:
# Add contextual features the meta-model doesn't see

# 1. Pair identity (label-encoded)
pair_map = {p: i for i, p in enumerate(sorted(oof_df['pair'].unique()))}
rejected['pair_id'] = rejected['pair'].map(pair_map)

# 2. Hour-of-day and day-of-week (cyclical encoding)
rejected['hour'] = rejected.index.hour
rejected['hour_sin'] = np.sin(2 * np.pi * rejected['hour'] / 24)
rejected['hour_cos'] = np.cos(2 * np.pi * rejected['hour'] / 24)
rejected['dow'] = rejected.index.dayofweek
rejected['dow_sin'] = np.sin(2 * np.pi * rejected['dow'] / 5)  # 5 trading days
rejected['dow_cos'] = np.cos(2 * np.pi * rejected['dow'] / 5)

# 3. Cross-pair signal agreement at same timestamp
# For each hour: how many pairs signal same direction as this pair's Q50?
q50_dirs = oof_df[['q50_dir', 'pair']].copy()
q50_dirs['ts'] = q50_dirs.index

# Count pairs with positive / negative Q50 at each timestamp
ts_counts = q50_dirs.groupby(q50_dirs.index).agg(
    n_positive=('q50_dir', lambda x: (x > 0).sum()),
    n_negative=('q50_dir', lambda x: (x < 0).sum()),
    n_pairs=('q50_dir', 'count')
)

rejected = rejected.join(ts_counts, how='left')
# Agreement = fraction of pairs agreeing with this pair's direction
rejected['cross_pair_agree'] = np.where(
    rejected['q50_dir'] > 0,
    rejected['n_positive'] / rejected['n_pairs'],
    rejected['n_negative'] / rejected['n_pairs']
)
rejected['cross_pair_agree'] = rejected['cross_pair_agree'].fillna(0)

# 4. Meta-model probability as continuous feature (already have it)
# 5. Whether this was Case A or Case B
rejected['is_tradeable_zone'] = (rejected['abs_Q50'] > MIN_Q50_THRESHOLD).astype(int)

# Define rescue feature columns
context_cols = [
    'pair_id', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'cross_pair_agree', 'n_positive', 'n_negative', 'n_pairs',
    'meta_proba', 'is_tradeable_zone'
]
rescue_feature_cols = meta_feature_cols + context_cols

print(f'Rescue features: {len(rescue_feature_cols)} ({len(meta_feature_cols)} original + {len(context_cols)} contextual)')
print(f'\nContextual features added:')
for c in context_cols:
    vals = rejected[c]
    print(f'  {c}: mean={vals.mean():.3f}, std={vals.std():.3f}, min={vals.min():.3f}, max={vals.max():.3f}')

## 3. Analyze Rejected Population

Before training, understand what we're working with. Is there structure in the rejected signals?

In [ ]:
# Analyze rejected signals by contextual dimensions
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
fig.patch.set_facecolor('#080c14')

def style_ax(ax):
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white', labelsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

# 1. WR by pair
ax = axes[0, 0]
style_ax(ax)
pair_wr = rejected.groupby('pair')['q50_correct'].agg(['mean', 'count'])
pair_wr = pair_wr.sort_values('mean', ascending=True)
colors = ['#4fc3f7' if wr > 0.55 else '#ff6b81' for wr in pair_wr['mean']]
bars = ax.barh(pair_wr.index, pair_wr['mean'], color=colors, alpha=0.8)
ax.axvline(0.5, color='white', linestyle='--', alpha=0.3)
ax.set_title('Rejected WR by Pair', color='white', fontsize=10)
ax.set_xlabel('Win Rate', color='white')
# Add count labels
for bar, (_, row) in zip(bars, pair_wr.iterrows()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'n={int(row["count"]):,}', color='white', va='center', fontsize=7)

# 2. WR by hour of day
ax = axes[0, 1]
style_ax(ax)
hour_wr = rejected.groupby('hour')['q50_correct'].mean()
ax.bar(hour_wr.index, hour_wr.values, color='#4fc3f7', alpha=0.8)
ax.axhline(0.5, color='white', linestyle='--', alpha=0.3)
ax.set_title('Rejected WR by Hour', color='white', fontsize=10)
ax.set_xlabel('Hour (UTC)', color='white')
ax.set_ylabel('Win Rate', color='white')

# 3. WR by day of week
ax = axes[0, 2]
style_ax(ax)
dow_wr = rejected.groupby('dow')['q50_correct'].mean()
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
ax.bar(range(len(dow_wr)), dow_wr.values, color='#4fc3f7', alpha=0.8)
ax.set_xticks(range(len(dow_labels)))
ax.set_xticklabels(dow_labels)
ax.axhline(0.5, color='white', linestyle='--', alpha=0.3)
ax.set_title('Rejected WR by Day', color='white', fontsize=10)
ax.set_ylabel('Win Rate', color='white')

# 4. WR by cross-pair agreement
ax = axes[1, 0]
style_ax(ax)
agree_bins = pd.cut(rejected['cross_pair_agree'], bins=5)
agree_wr = rejected.groupby(agree_bins, observed=True)['q50_correct'].agg(['mean', 'count'])
ax.bar(range(len(agree_wr)), agree_wr['mean'], color='#4fc3f7', alpha=0.8)
ax.set_xticks(range(len(agree_wr)))
ax.set_xticklabels([f'{x.left:.1%}-{x.right:.1%}' for x in agree_wr.index], rotation=30, fontsize=7)
ax.axhline(0.5, color='white', linestyle='--', alpha=0.3)
ax.set_title('Rejected WR by Cross-Pair Agreement', color='white', fontsize=10)
ax.set_ylabel('Win Rate', color='white')

# 5. WR by meta_proba band (Case A only)
ax = axes[1, 1]
style_ax(ax)
case_a = rejected[rejected['is_tradeable_zone'] == 1]
if len(case_a) > 0:
    meta_bins = pd.cut(case_a['meta_proba'], bins=[0, 0.30, 0.40, 0.45, 0.50, 0.55])
    meta_wr = case_a.groupby(meta_bins, observed=True)['q50_correct'].agg(['mean', 'count'])
    ax.bar(range(len(meta_wr)), meta_wr['mean'], color='#ff6b81', alpha=0.8)
    ax.set_xticks(range(len(meta_wr)))
    ax.set_xticklabels([str(x) for x in meta_wr.index], rotation=30, fontsize=7)
    for i, (_, row) in enumerate(meta_wr.iterrows()):
        ax.text(i, row['mean'] + 0.01, f'n={int(row["count"]):,}', color='white', ha='center', fontsize=7)
ax.axhline(0.5, color='white', linestyle='--', alpha=0.3)
ax.set_title('Case A: WR by Meta Proba Band', color='white', fontsize=10)
ax.set_ylabel('Win Rate', color='white')

# 6. Case A vs Case B
ax = axes[1, 2]
style_ax(ax)
case_stats = rejected.groupby('is_tradeable_zone')['q50_correct'].agg(['mean', 'count'])
labels = ['Case B\n(|Q50|<0.5x)', 'Case A\n(meta<0.55)']
colors = ['#ff6b81', '#4fc3f7']
bars = ax.bar(range(len(case_stats)), case_stats['mean'], color=colors, alpha=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.axhline(0.5, color='white', linestyle='--', alpha=0.3)
for i, (_, row) in enumerate(case_stats.iterrows()):
    ax.text(i, row['mean'] + 0.01, f'n={int(row["count"]):,}\nWR={row["mean"]:.1%}', 
            color='white', ha='center', fontsize=9)
ax.set_title('Case A vs Case B', color='white', fontsize=10)
ax.set_ylabel('Win Rate', color='white')

plt.suptitle('Rejected Signal Analysis — Is There Structure?', color='white', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Train Rescue Model (Walk-Forward CV)

Binary classifier trained **only on rejected signals**: predicts P(Q50 direction is correct) for signals the current pipeline would skip.

Walk-forward CV to avoid leakage — same discipline as meta-model.

In [ ]:
# Rescue model parameters — conservative to avoid overfitting on noisy rejected population
rescue_params = {
    'objective':         'binary',
    'metric':            'auc',
    'boosting_type':     'gbdt',
    'n_estimators':      3000,
    'learning_rate':     0.01,
    'num_leaves':        32,
    'max_depth':         4,
    'min_child_samples': 50,   # higher than meta-model — more regularization
    'feature_fraction':  0.5,
    'bagging_fraction':  0.7,
    'bagging_freq':      5,
    'reg_alpha':         1.0,  # stronger L1
    'reg_lambda':        1.0,  # stronger L2
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
    'device':            'gpu',
    'is_unbalance':      True,
}

# Prepare rescue training data
X_rescue = rejected[rescue_feature_cols].ffill().fillna(0)
y_rescue = rejected['q50_correct']

rescue_splits = walk_forward_splits(len(X_rescue), n_splits=5, test_ratio=0.1)
rescue_oof_proba = np.full(len(X_rescue), np.nan)
rescue_best_iters = []
rescue_aucs = []

print(f'Rescue model training: {len(X_rescue):,} rejected rows')
print(f'Class balance: {y_rescue.mean():.1%} correct\n')

for fold, (train_idx, test_idx) in enumerate(rescue_splits):
    X_tr, y_tr = X_rescue.iloc[train_idx], y_rescue.iloc[train_idx]
    X_te, y_te = X_rescue.iloc[test_idx], y_rescue.iloc[test_idx]
    
    model = lgb.LGBMClassifier(**rescue_params)
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    
    proba = model.predict_proba(X_te)[:, 1]
    rescue_oof_proba[test_idx] = proba
    
    auc = roc_auc_score(y_te, proba)
    rescue_aucs.append(auc)
    rescue_best_iters.append(model.best_iteration_)
    
    print(f'  Fold {fold+1}: AUC={auc:.4f}, best_iter={model.best_iteration_}, '
          f'train_acc={y_tr.mean():.1%}, test_acc={y_te.mean():.1%}')

del model; gc.collect()

print(f'\nMean CV AUC: {np.mean(rescue_aucs):.4f}')
print(f'Avg best iterations: {int(np.mean(rescue_best_iters))}')

## 5. OOF Rescue Model Evaluation

Sweep rescue probability thresholds. How many signals can we rescue at what WR?

In [ ]:
# Evaluate rescue model OOF at different probability thresholds
has_rescue_oof = ~np.isnan(rescue_oof_proba)
rescue_eval = rejected[has_rescue_oof].copy()
rescue_eval['rescue_proba'] = rescue_oof_proba[has_rescue_oof]

print(f'OOF rows with rescue predictions: {len(rescue_eval):,}')
print(f'\n{"Threshold":<12} {"Rescued":>8} {"WR":>8} {"EV/trade":>12} {"TotalPnL":>10}')
print('-' * 55)

rescue_sweep_oof = []
for thresh in np.arange(0.50, 0.96, 0.05):
    mask = rescue_eval['rescue_proba'] > thresh
    n = mask.sum()
    if n < 10:
        continue
    s = rescue_eval[mask]
    pnl = s['q50_dir'] * s['actual_return'] - AVG_SPREAD
    wr = s['q50_correct'].mean()
    rescue_sweep_oof.append({'thresh': thresh, 'n': n, 'wr': wr, 'ev': pnl.mean(), 'pnl': pnl.sum()})
    flag = ' <<<' if wr >= 0.65 and n >= 100 else ''
    print(f'R > {thresh:.2f}    {n:>8,} {wr:>7.1%} {pnl.mean():>12.6f} {pnl.sum():>10.4f}{flag}')

## 6. Train Final Rescue Model & Test Set Evaluation

Train final rescue model on all rejected training rows. Evaluate on genuinely unseen test set.
Then combine with existing pipeline: meta P>0.55 trades + rescued trades.

In [ ]:
# Train final rescue model on all rejected training rows
avg_iter = max(50, int(np.mean(rescue_best_iters)))
final_rescue = lgb.LGBMClassifier(**{**rescue_params, 'n_estimators': avg_iter})
final_rescue.fit(X_rescue, y_rescue)

print(f'Final rescue model trained: {avg_iter} iterations on {len(X_rescue):,} rows')

# Save rescue model
joblib.dump({
    'model': final_rescue,
    'rescue_feature_cols': rescue_feature_cols,
    'context_cols': context_cols,
    'meta_feature_cols': meta_feature_cols,
    'pair_map': pair_map,
    'train_end': TRAIN_END,
    'cv_auc': np.mean(rescue_aucs),
    'n_iters': avg_iter,
}, RESCUE_DIR / 'rescue_model.joblib')

size_mb = (RESCUE_DIR / 'rescue_model.joblib').stat().st_size / 1024 / 1024
print(f'Saved to {RESCUE_DIR / "rescue_model.joblib"} ({size_mb:.1f} MB)')

In [ ]:
# --- Test Set Evaluation ---
# Step 1: Get Q25/Q50/Q75 predictions on test set
X_test = df_test[feature_cols].ffill().fillna(0)
y_test = df_test[TARGET_COL]
valid_test = y_test.notna()
X_test_clean = X_test[valid_test]
y_test_clean = y_test[valid_test]

q_preds_test = {}
for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
    q_int = int(q * 100)
    bundle = joblib.load(MODELS_DIR / f'model_1H_Q{q_int}.joblib')
    q_preds_test[q_name] = bundle['model'].predict(X_test_clean)

# Build test results
test_results = X_test_clean.copy()
test_results['Q25_oof'] = q_preds_test['Q25']
test_results['Q50_oof'] = q_preds_test['Q50']
test_results['Q75_oof'] = q_preds_test['Q75']
test_results['abs_Q50'] = np.abs(test_results['Q50_oof'])
test_results['iqr'] = test_results['Q75_oof'] - test_results['Q25_oof']
test_results['conf_ratio'] = test_results['abs_Q50'] / test_results['iqr'].clip(lower=1e-10)
test_results['pred_dir'] = np.sign(test_results['Q50_oof'])
test_results['actual_return'] = y_test_clean.values
test_results['actual_dir'] = np.sign(test_results['actual_return'])
test_results['pair'] = df_test.loc[valid_test, 'pair'].values
test_results['q50_correct'] = (test_results['pred_dir'] == test_results['actual_dir']).astype(int)

# Step 2: Apply meta-model to tradeable zone
meta_bundle = joblib.load(META_DIR / 'meta_confidence.joblib')
meta_model = meta_bundle['model']

test_tradeable = test_results[test_results['abs_Q50'] > MIN_Q50_THRESHOLD].copy()
meta_proba_test = meta_model.predict_proba(test_tradeable[meta_feature_cols])[:, 1]
test_tradeable['meta_proba'] = meta_proba_test

# Assign meta_proba back to full test_results
test_results['meta_proba'] = np.nan
test_results.loc[test_tradeable.index, 'meta_proba'] = meta_proba_test

# Step 3: Identify rejected test signals
test_accepted = (test_results['abs_Q50'] > MIN_Q50_THRESHOLD) & (test_results['meta_proba'] > META_THRESHOLD)
test_rejected = test_results[~test_accepted].copy()
test_rejected['meta_proba'] = test_rejected['meta_proba'].fillna(0)

print(f'Test set total: {len(test_results):,}')
print(f'Test accepted (meta P>{META_THRESHOLD}): {test_accepted.sum():,}')
print(f'Test rejected (rescue candidates): {len(test_rejected):,}')

n_test_days = (test_results.index.max() - test_results.index.min()).days
print(f'Test period: {n_test_days} days')

In [ ]:
# Step 4: Add contextual features to test rejected signals
test_rejected['pair_id'] = test_rejected['pair'].map(pair_map)
test_rejected['hour'] = test_rejected.index.hour
test_rejected['hour_sin'] = np.sin(2 * np.pi * test_rejected['hour'] / 24)
test_rejected['hour_cos'] = np.cos(2 * np.pi * test_rejected['hour'] / 24)
test_rejected['dow'] = test_rejected.index.dayofweek
test_rejected['dow_sin'] = np.sin(2 * np.pi * test_rejected['dow'] / 5)
test_rejected['dow_cos'] = np.cos(2 * np.pi * test_rejected['dow'] / 5)

# Cross-pair agreement on test set
test_q50_dirs = test_results[['pred_dir', 'pair']].copy()
test_ts_counts = test_q50_dirs.groupby(test_q50_dirs.index).agg(
    n_positive=('pred_dir', lambda x: (x > 0).sum()),
    n_negative=('pred_dir', lambda x: (x < 0).sum()),
    n_pairs=('pred_dir', 'count')
)
test_rejected = test_rejected.join(test_ts_counts, how='left')
test_rejected['cross_pair_agree'] = np.where(
    test_rejected['pred_dir'] > 0,
    test_rejected['n_positive'] / test_rejected['n_pairs'],
    test_rejected['n_negative'] / test_rejected['n_pairs']
)
test_rejected['cross_pair_agree'] = test_rejected['cross_pair_agree'].fillna(0)
test_rejected['is_tradeable_zone'] = (test_rejected['abs_Q50'] > MIN_Q50_THRESHOLD).astype(int)

# Step 5: Apply rescue model
rescue_proba_test = final_rescue.predict_proba(test_rejected[rescue_feature_cols].ffill().fillna(0))[:, 1]
test_rejected['rescue_proba'] = rescue_proba_test

print(f'Rescue model probability distribution on test:')
print(f'  mean={rescue_proba_test.mean():.3f}, median={np.median(rescue_proba_test):.3f}')
print(f'  min={rescue_proba_test.min():.3f}, max={rescue_proba_test.max():.3f}')

In [ ]:
# Sweep rescue thresholds on TEST set — rescued signals only
print(f'RESCUE MODEL TEST SET RESULTS (unseen, post {TRAIN_END})')
print(f'\n{"Threshold":<12} {"Rescued":>8} {"R/day":>8} {"WR":>8} {"EV/trade":>12} {"TotalPnL":>10} {"Sharpe":>8}')
print('-' * 75)

rescue_sweep_test = []
for thresh in np.arange(0.50, 0.96, 0.05):
    mask = test_rejected['rescue_proba'] > thresh
    n = mask.sum()
    if n < 10:
        continue
    s = test_rejected[mask]
    pnl = s['pred_dir'] * s['actual_return'] - AVG_SPREAD
    wr = s['q50_correct'].mean()
    ev = pnl.mean()
    total_pnl = pnl.sum()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    daily = n / n_test_days
    
    rescue_sweep_test.append({'thresh': thresh, 'trades': n, 'daily': daily, 'wr': wr, 'ev': ev, 'pnl': total_pnl, 'sharpe': sharpe})
    flag = ' <<<' if wr >= 0.65 and n >= 100 else (' <<' if wr >= 0.60 else '')
    print(f'R > {thresh:.2f}    {n:>8,} {daily:>8.2f} {wr:>7.1%} {ev:>12.6f} {total_pnl:>10.4f} {sharpe:>8.2f}{flag}')

# Baseline: all rejected signals without rescue model
print(f'\n--- Baseline: all rejected (no rescue model) ---')
pnl_all = test_rejected['pred_dir'] * test_rejected['actual_return'] - AVG_SPREAD
wr_all = test_rejected['q50_correct'].mean()
sharpe_all = (pnl_all.mean() / pnl_all.std()) * np.sqrt(252 * 24) if pnl_all.std() > 0 else 0
print(f'{"All rejected":<12} {len(test_rejected):>8,} {len(test_rejected)/n_test_days:>8.2f} {wr_all:>7.1%} {pnl_all.mean():>12.6f} {pnl_all.sum():>10.4f} {sharpe_all:>8.2f}')

## 7. Combined Pipeline — Meta + Rescue

The key evaluation: existing pipeline (meta P>0.55) + rescued signals combined.
Is the total PnL higher than meta-only?

In [ ]:
# Combined evaluation: meta P>0.55 trades + rescued trades at various thresholds
meta_accepted_test = test_tradeable[test_tradeable['meta_proba'] > META_THRESHOLD].copy()
meta_pnl = meta_accepted_test['pred_dir'] * meta_accepted_test['actual_return'] - AVG_SPREAD
meta_wr = (meta_accepted_test['pred_dir'] == meta_accepted_test['actual_dir']).mean()

print(f'COMBINED PIPELINE RESULTS (Test Set)')
print(f'\nBaseline — Meta P>{META_THRESHOLD} only:')
print(f'  Trades: {len(meta_accepted_test):,} ({len(meta_accepted_test)/n_test_days:.2f}/day)')
print(f'  WR: {meta_wr:.1%}, EV: {meta_pnl.mean():.6f}, PnL: {meta_pnl.sum():.4f}')

print(f'\n{"Rescue Thresh":<14} {"Meta Tr":>8} {"Rescued":>8} {"Total":>8} {"Tr/day":>8} {"WR":>8} {"EV/trade":>12} {"TotalPnL":>10} {"vs Meta":>10}')
print('-' * 100)

combined_results = []
for r in rescue_sweep_test:
    rescued = test_rejected[test_rejected['rescue_proba'] > r['thresh']]
    rescued_pnl = rescued['pred_dir'] * rescued['actual_return'] - AVG_SPREAD
    
    # Combined
    total_trades = len(meta_accepted_test) + len(rescued)
    total_pnl_val = meta_pnl.sum() + rescued_pnl.sum()
    combined_pnl_series = pd.concat([meta_pnl, rescued_pnl])
    total_wr_num = (meta_accepted_test['pred_dir'] == meta_accepted_test['actual_dir']).sum() + rescued['q50_correct'].sum()
    total_wr = total_wr_num / total_trades
    total_ev = combined_pnl_series.mean()
    sharpe = (combined_pnl_series.mean() / combined_pnl_series.std()) * np.sqrt(252 * 24) if combined_pnl_series.std() > 0 else 0
    delta = total_pnl_val - meta_pnl.sum()
    
    combined_results.append({
        'rescue_thresh': r['thresh'], 'meta_trades': len(meta_accepted_test),
        'rescued': len(rescued), 'total': total_trades, 'wr': total_wr,
        'ev': total_ev, 'pnl': total_pnl_val, 'delta': delta, 'sharpe': sharpe
    })
    
    flag = ' <<<' if delta > 0 else ''
    print(f'R > {r["thresh"]:.2f}      {len(meta_accepted_test):>8,} {len(rescued):>8,} {total_trades:>8,} '
          f'{total_trades/n_test_days:>8.2f} {total_wr:>7.1%} {total_ev:>12.6f} {total_pnl_val:>10.4f} '
          f'{delta:>+10.4f}{flag}')

# Find best combined config
if combined_results:
    best = max(combined_results, key=lambda x: x['pnl'])
    print(f'\nBest combined config: rescue R>{best["rescue_thresh"]:.2f}')
    print(f'  +{best["rescued"]} rescued trades, total PnL: {best["pnl"]:.4f} (delta: {best["delta"]:+.4f})')
    print(f'  PnL improvement: {best["delta"]/meta_pnl.sum()*100:+.1f}%')

## 8. Equity Curves — Meta-Only vs Combined

In [ ]:
# Pick best rescue threshold (highest combined PnL with positive delta)
positive_deltas = [r for r in combined_results if r['delta'] > 0]
if positive_deltas:
    best_rescue = max(positive_deltas, key=lambda x: x['pnl'])
    RESCUE_THRESH = best_rescue['rescue_thresh']
else:
    RESCUE_THRESH = 0.65  # fallback
    print('WARNING: No rescue threshold improved PnL. Using R>0.65 for visualization.')

print(f'Selected rescue threshold: R > {RESCUE_THRESH:.2f}')

rescued_test = test_rejected[test_rejected['rescue_proba'] > RESCUE_THRESH]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#080c14')

# 1. Meta-only equity curve
ax = axes[0]
ax.set_facecolor('#080c14')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

meta_eq = (meta_accepted_test['pred_dir'] * meta_accepted_test['actual_return'] - AVG_SPREAD).cumsum()
ax.plot(meta_eq.index, meta_eq.values, color='#4fc3f7', linewidth=2)
ax.axhline(0, color=(1,1,1,0.2), linewidth=1, linestyle='--')
ax.set_title(f'Meta P>{META_THRESHOLD} Only\n{len(meta_accepted_test):,} tr, WR:{meta_wr:.1%}, PnL:{meta_pnl.sum():.2f}', 
             color='white', fontsize=10)
ax.set_ylabel('Cum PnL', color='white')

# 2. Rescued-only equity curve
ax = axes[1]
ax.set_facecolor('#080c14')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

rescued_pnl_series = rescued_test['pred_dir'] * rescued_test['actual_return'] - AVG_SPREAD
rescued_eq = rescued_pnl_series.cumsum()
rescued_wr = rescued_test['q50_correct'].mean()
ax.plot(rescued_eq.index, rescued_eq.values, color='#ff6b81', linewidth=2)
ax.axhline(0, color=(1,1,1,0.2), linewidth=1, linestyle='--')
ax.set_title(f'Rescued R>{RESCUE_THRESH:.2f} Only\n{len(rescued_test):,} tr, WR:{rescued_wr:.1%}, PnL:{rescued_pnl_series.sum():.2f}', 
             color='white', fontsize=10)
ax.set_ylabel('Cum PnL', color='white')

# 3. Combined equity curve
ax = axes[2]
ax.set_facecolor('#080c14')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

# Merge meta + rescued, sort by timestamp
combined_trades = pd.concat([
    pd.DataFrame({'pnl': meta_pnl, 'source': 'meta'}, index=meta_accepted_test.index),
    pd.DataFrame({'pnl': rescued_pnl_series, 'source': 'rescue'}, index=rescued_test.index),
]).sort_index()
combined_eq = combined_trades['pnl'].cumsum()

# Also plot meta-only for comparison
ax.plot(meta_eq.index, meta_eq.values, color='#4fc3f7', linewidth=1, alpha=0.5, label='Meta only')
ax.plot(combined_eq.index, combined_eq.values, color='#66bb6a', linewidth=2, label='Meta + Rescue')
ax.axhline(0, color=(1,1,1,0.2), linewidth=1, linestyle='--')
total_n = len(meta_accepted_test) + len(rescued_test)
total_pnl = meta_pnl.sum() + rescued_pnl_series.sum()
ax.set_title(f'Combined (Meta + Rescue)\n{total_n:,} tr, PnL:{total_pnl:.2f} (delta:{total_pnl-meta_pnl.sum():+.2f})', 
             color='white', fontsize=10)
ax.set_ylabel('Cum PnL', color='white')
ax.legend(fontsize=8, facecolor='#080c14', edgecolor='#1a2332', labelcolor='white')

plt.suptitle('Meta-Only vs Combined Pipeline — Equity Curves (Test Set)', color='white', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Rescue Model Feature Importance

What does the rescue model use to identify correct signals among the rejected population?

In [ ]:
importance = pd.Series(final_rescue.feature_importances_, index=rescue_feature_cols)
importance = importance.sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.patch.set_facecolor('#080c14')

for ax in axes:
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white', labelsize=7)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

# Top 30 features
top30 = importance.tail(30)
colors = ['#ff6b81' if f in context_cols else '#4fc3f7' for f in top30.index]
axes[0].barh(top30.index, top30.values, color=colors, alpha=0.8)
axes[0].set_title('Top 30 Features (red = contextual)', color='white', fontsize=11)

# Contextual features specifically
ctx_imp = importance[context_cols].sort_values(ascending=True)
axes[1].barh(ctx_imp.index, ctx_imp.values, color='#ff6b81', alpha=0.8)
axes[1].set_title('Contextual Features Only', color='white', fontsize=11)

plt.suptitle('Rescue Model Feature Importance — What Drives Signal Recovery?', color='white', fontsize=13)
plt.tight_layout()
plt.show()

# Print top 10
print('\nTop 10 rescue model features:')
for i, (feat, imp) in enumerate(importance.tail(10).iloc[::-1].items(), 1):
    tag = ' (CONTEXTUAL)' if feat in context_cols else ''
    print(f'  {i}. {feat}: {imp}{tag}')

## 10. Per-Pair Breakdown — Rescued Signals

In [ ]:
# Per-pair breakdown of rescued signals at selected threshold
print(f'Rescued signals at R > {RESCUE_THRESH:.2f} — Per-Pair Breakdown (Test Set)')
print(f'\n{"Pair":<10} {"Rescued":>8} {"R/day":>8} {"WR":>8} {"EV/trade":>12} {"TotalPnL":>10}')
print('-' * 60)

for pair in sorted(rescued_test['pair'].unique()):
    p = rescued_test[rescued_test['pair'] == pair]
    pnl = p['pred_dir'] * p['actual_return'] - AVG_SPREAD
    wr = p['q50_correct'].mean()
    flag = ' <<<' if pnl.mean() > 0 else ''
    print(f'{pair:<10} {len(p):>8,} {len(p)/n_test_days:>8.2f} {wr:>7.1%} {pnl.mean():>12.6f} {pnl.sum():>10.4f}{flag}')

## 11. Summary

In [ ]:
print('=' * 80)
print('RESCUE MODEL — SUMMARY')
print('=' * 80)

print(f'\nRescue model: LightGBM binary classifier')
print(f'  CV AUC: {np.mean(rescue_aucs):.4f}')
print(f'  Features: {len(rescue_feature_cols)} ({len(meta_feature_cols)} meta + {len(context_cols)} contextual)')
print(f'  Contextual features: {context_cols}')
print(f'  Trained on: rejected signals (meta P<{META_THRESHOLD} or |Q50|<0.5x) before {TRAIN_END}')
print(f'  Saved to: {RESCUE_DIR / "rescue_model.joblib"}')

print(f'\n--- Pipeline Comparison (Test Set, {n_test_days} days) ---')
print(f'{"Strategy":<30} {"Trades":>8} {"Tr/day":>8} {"WR":>8} {"EV":>12} {"PnL":>10}')
print('-' * 80)

# Meta-only
meta_label = 'Meta P>0.55 (current)'
print(f'{meta_label:<30} {len(meta_accepted_test):>8,} {len(meta_accepted_test)/n_test_days:>8.2f} '
      f'{meta_wr:>7.1%} {meta_pnl.mean():>12.6f} {meta_pnl.sum():>10.4f}')

# Best combined
if positive_deltas:
    best = max(positive_deltas, key=lambda x: x['pnl'])
    combo_label = f'Meta + Rescue R>{best["rescue_thresh"]:.2f}'
    print(f'{combo_label:<30} {best["total"]:>8,} {best["total"]/n_test_days:>8.2f} '
          f'{best["wr"]:>7.1%} {best["ev"]:>12.6f} {best["pnl"]:>10.4f}')
    print(f'\nDelta PnL: {best["delta"]:+.4f} ({best["delta"]/meta_pnl.sum()*100:+.1f}%)')
    print(f'Extra trades rescued: {best["rescued"]:,}')
else:
    print(f'\nNo rescue threshold produced positive delta PnL.')
    print(f'The rescue model may need:')
    print(f'  - More contextual features')
    print(f'  - Different model architecture')
    print(f'  - Or the rejected population truly has no recoverable structure')

print(f'\n{"=" * 80}')